# 04 – Train / Test Split

**Proyecto:** Predicción de Subempleo por Insuficiencia de Horas — EPEN 2024  
**Etapa:** División del dataset para modelado  
**Dataset de entrada:** `data/feature_engineering/epen_features_final.csv`  
**Datasets de salida:** `X_train.csv`, `X_test.csv`, `y_train.csv`, `y_test.csv`

## Objetivo

Dividir el dataset final de feature engineering en conjuntos de **entrenamiento (80%)** y **prueba (20%)** con estratificación sobre el target `target_subempleo_horas`.

### Decisiones metodológicas

| Decisión | Valor | Justificación |
|:---------|:------|:--------------|
| Test size | 20% | Suficiente para evaluación con ~24k registros |
| Estratificación | `target_subempleo_horas` | Preserva la proporción del target (~25% / ~75%) en ambos conjuntos |
| random_state | 42 | Reproducibilidad |

### Flujo del pipeline

```
epen_features_final.csv
        │
        ▼
   train_test_split  (este notebook)
        │
   ┌────┴────┐
   │         │
X_train   X_test
y_train   y_test
   │
   ▼
05_feature_selection/  ← selección estadística SOLO sobre X_train
   │
   ▼
06_modelling/
```

> ⚠️ La **selección estadística de features** debe realizarse **después** del split, únicamente sobre `X_train`, para evitar *data leakage*.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split

pd.set_option('display.max_columns', 80)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Cargar dataset final de feature engineering ──────────────────────────────
INPUT_PATH = Path('../data/feature_engineering/epen_features_final.csv')

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f'No se encontró: {INPUT_PATH}\n'
        'Ejecuta primero: 04_feature_engineering/04_income_features.ipynb'
    )

df = pd.read_csv(INPUT_PATH, low_memory=False)
print(f'Dataset cargado : {INPUT_PATH.name}')
print(f'Dimensiones     : {df.shape[0]:,} filas x {df.shape[1]} columnas')

In [ ]:
# ── Validación del dataset ────────────────────────────────────────────────────
assert 'target_subempleo_horas' in df.columns, \
    "ERROR: 'target_subempleo_horas' no encontrado en el dataset."
assert df['target_subempleo_horas'].isnull().sum() == 0, \
    'ERROR: target_subempleo_horas tiene valores nulos.'
print('target_subempleo_horas presente y sin nulos: OK')

for lv in ['P209H', 'C333', 'C334']:
    assert lv not in df.columns, f'ERROR: variable de leakage {lv} presente.'
print('Variables de leakage ausentes: OK')

print('\nDistribución del target (dataset completo):')
counts = df['target_subempleo_horas'].value_counts().sort_index()
pct    = df['target_subempleo_horas'].value_counts(normalize=True).sort_index() * 100
display(pd.DataFrame({'conteo': counts, 'porcentaje (%)': pct.round(2)}))

---
## 1. Separar Features y Target

In [ ]:
TARGET = 'target_subempleo_horas'

# Columnas a excluir de X (target y variables de leakage)
excluir = [TARGET, 'fa_son24']
excluir = [c for c in excluir if c in df.columns]

X = df.drop(columns=excluir)
y = df[TARGET]

print(f'X : {X.shape[0]:,} filas x {X.shape[1]} columnas')
print(f'y : {y.shape[0]:,} observaciones')

---
## 2. División Estratificada 80/20

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print('División completada:')
print(f'  X_train : {X_train.shape[0]:>7,} filas x {X_train.shape[1]} columnas')
print(f'  X_test  : {X_test.shape[0]:>7,} filas x {X_test.shape[1]} columnas')
print(f'  y_train : {y_train.shape[0]:>7,} observaciones')
print(f'  y_test  : {y_test.shape[0]:>7,} observaciones')

---
## 3. Verificar Distribución del Target

In [ ]:
print('Distribución del target en cada conjunto:')
resumen = pd.DataFrame({
    'total'  : y.value_counts().sort_index(),
    'train'  : y_train.value_counts().sort_index(),
    'test'   : y_test.value_counts().sort_index(),
})
resumen['pct_total (%)'] = (resumen['total']  / resumen['total'].sum()  * 100).round(2)
resumen['pct_train (%)'] = (resumen['train']  / resumen['train'].sum()  * 100).round(2)
resumen['pct_test (%)']  = (resumen['test']   / resumen['test'].sum()   * 100).round(2)
display(resumen)

---
## 4. Guardar Conjuntos de Datos

In [ ]:
OUTPUT_DIR = Path('../data/split')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

X_train.to_csv(OUTPUT_DIR / 'X_train.csv', index=False)
X_test.to_csv( OUTPUT_DIR / 'X_test.csv',  index=False)
y_train.to_csv(OUTPUT_DIR / 'y_train.csv', index=False)
y_test.to_csv( OUTPUT_DIR / 'y_test.csv',  index=False)

print('Archivos guardados en:', OUTPUT_DIR)
for fname, obj in [('X_train.csv', X_train), ('X_test.csv', X_test),
                   ('y_train.csv', y_train), ('y_test.csv',  y_test)]:
    print(f'  {fname:<15}: {obj.shape}')

---
## 5. Nota sobre Feature Selection

> ⚠️ **Importante para los siguientes pasos del pipeline:**
>
> La **selección estadística de variables** (correlación, importancia, tests de hipótesis, etc.) debe realizarse **exclusivamente sobre `X_train`**, nunca sobre el dataset completo ni sobre `X_test`.
>
> Hacerlo sobre el dataset completo introduciría **data leakage**: el modelo "sabría" información de los datos de prueba antes de ser evaluado.
>
> El flujo correcto es:
> 1. **Feature engineering** conceptual (reglas fijas) → puede hacerse antes del split ✅
> 2. **Train/test split** (este notebook) ✅
> 3. **Feature selection** estadística → solo sobre `X_train` ← siguiente etapa
> 4. **Balanceo** (SMOTE, undersampling) → solo sobre el conjunto de entrenamiento
> 5. **Entrenamiento del modelo** → sobre X_train balanceado
> 6. **Evaluación** → sobre X_test (nunca visto durante entrenamiento)